# D19 S2 A1 - Transfer Learning Parameter Budgets


In [1]:
backbone = [
    {"type": "conv", "kh": 3, "kw": 3, "in_ch": 3,   "out_ch": 32},
    {"type": "pool"},
    {"type": "conv", "kh": 3, "kw": 3, "in_ch": 32,  "out_ch": 64},
    {"type": "pool"},
    {"type": "conv", "kh": 3, "kw": 3, "in_ch": 64,  "out_ch": 128},
    {"type": "pool"},
    {"type": "conv", "kh": 3, "kw": 3, "in_ch": 128, "out_ch": 128},
    {"type": "pool"},
]
head = [
    {"type": "flatten"},
    {"type": "dense", "in": 128, "out": 128},
    {"type": "dense", "in": 128, "out": 10},
]


In [2]:
def layer_params(layer):
    if layer["type"] == "conv":
        return (layer["kh"] * layer["kw"] * layer["in_ch"] + 1) * layer["out_ch"]
    if layer["type"] == "dense":
        return (layer["in"] + 1) * layer["out"]
    return 0


def count_params(layers):
    return sum(layer_params(layer) for layer in layers)


def build_transfer_model(backbone, head, freeze_backbone=True):
    backbone_params = count_params(backbone)
    head_params = count_params(head)
    total = backbone_params + head_params
    if freeze_backbone:
        return {"trainable": head_params, "frozen": backbone_params, "total": total}
    return {"trainable": total, "frozen": 0, "total": total}


def trainable_ratio(model):
    if model["total"] == 0:
        return 0.0
    return model["trainable"] / model["total"]


def finetune_topk(backbone, head, k):
    backbone_params = count_params(backbone)
    head_params = count_params(head)
    total = backbone_params + head_params
    param_layers = [layer_params(layer) for layer in backbone if layer_params(layer) > 0]
    if k <= 0:
        unfrozen_backbone = 0
    else:
        unfrozen_backbone = sum(param_layers[-k:])
    trainable = head_params + unfrozen_backbone
    return {"trainable": trainable, "frozen": total - trainable, "total": total}


def strategy_budgets(backbone, head):
    feature = build_transfer_model(backbone, head, True)["trainable"]
    top2 = finetune_topk(backbone, head, 2)["trainable"]
    full = build_transfer_model(backbone, head, False)["trainable"]
    return {"feature_extraction": feature, "finetune_topk2": top2, "full": full}


## Verification


In [3]:
m = build_transfer_model(backbone, head, freeze_backbone=True)
print(m)
print(round(trainable_ratio(m), 3))
print(build_transfer_model(backbone, head, False))
print(finetune_topk(backbone, head, 1))
print(strategy_budgets(backbone, head))


## Autograder-style checks


In [4]:
assert layer_params({"type":"conv","kh":3,"kw":3,"in_ch":3,"out_ch":32}) == 896
assert layer_params({"type":"dense","in":128,"out":10}) == 1290
assert layer_params({"type":"pool"}) == 0
assert count_params(backbone) == 240832
m = build_transfer_model(backbone, head, True)
assert m == {"trainable":17802,"frozen":240832,"total":258634}
assert build_transfer_model(backbone, head, False) == {"trainable":258634,"frozen":0,"total":258634}
assert abs(trainable_ratio(m) - 17802/258634) < 1e-12
assert finetune_topk(backbone, head, 1) == {"trainable":165386,"frozen":93248,"total":258634}
b = strategy_budgets(backbone, head)
assert b == {"feature_extraction":17802,"finetune_topk2":239242,"full":258634}
assert b["feature_extraction"] < b["finetune_topk2"] < b["full"]
assert finetune_topk(backbone, head, 0)["trainable"] == 17802
assert finetune_topk(backbone, head, 100)["trainable"] == 258634
assert trainable_ratio({"trainable":0,"frozen":0,"total":0}) == 0.0
print("All autograder-style checks passed.")
